# **1. CONTEXT**

# Bellabeat Case Study: Análise de Dispositivos Inteligentes

## 1. Visão Geral e Contexto
A Bellabeat é uma empresa de tecnologia pioneira no mercado de *wellness* feminino, unindo design elegante e monitoramento de saúde. Como analista deste estudo de caso, o desafio é transformar dados brutos de uso de dispositivos inteligentes (atividade, sono e hábitos diários) em recomendações estratégicas para a diretoria de marketing.

O foco central é entender como os consumidores utilizam dispositivos inteligentes para identificar oportunidades de posicionamento para o **Bellabeat Leaf**, o carro-chefe da marca, focado em bem-estar holístico e estilo de vida.

## 2. Objetivos Estratégicos
*   **Análise de Comportamento:** Mapear padrões de uso diário e flutuações de engajamento ao longo da semana.
*   **Otimização de Marketing:** Gerar insights que permitam à Bellabeat transitar de uma marca de hardware para uma parceira essencial na jornada de saúde da mulher.
*   **Identificação de Oportunidades:** Avaliar lacunas no engajamento para sugerir novas funcionalidades ou campanhas de retenção.

## 3. Metodologia e Escopo
*   **Dataset:** Dados de monitoramento de 35 usuários (*FitBit Fitness Tracker Data*).
*   **Abordagem:** Devido ao tamanho amostral ($n=35$), este projeto é tratado como um **estudo de prova de conceito (PoC)**. A análise foca em tendências qualitativas e padrões comportamentais que servem como hipóteses para validação em larga escala.
*   **Foco no Produto:** A análise prioriza insights para o **Bellabeat Leaf**, explorando sua versatilidade como um rastreador que monitora saúde de forma "invisível" e elegante.

---

## 4. Avaliação do Cenário e Insights Preliminares

### A. Dinâmica de Retenção e Fidelidade
A análise diferencia o **Uso Geral** do app do **Registro de Sono**. Essa distinção é crucial: enquanto o uso geral mede a retenção básica, o registro de sono indica uma adoção mais profunda do ecossistema de saúde.
> **Hipótese:** Usuárias com alta consistência no registro de sono possuem maior LTV (*Lifetime Value*) e são as candidatas ideais para o programa de assinatura Bellabeat+.

### B. Oportunidades Estratégicas
*   **Gamificação Direcionada:** Identificar quedas de atividade (ex: finais de semana) para implementar desafios de movimento via notificações push.
*   **Personalização Preditiva:** Cruzar dados para oferecer recomendações preventivas. Exemplo: *"Notamos que seu sono é mais restaurador após dias de atividade moderada. Que tal uma caminhada hoje?"*

---

## 5. Perguntas de Negócio Norteadoras
1.  Quais são as principais tendências de uso observadas nos dispositivos inteligentes?
2.  Como esses padrões de comportamento se traduzem em personas de consumo para a Bellabeat?
3.  De que forma esses insights podem refinar a alocação de orçamento de marketing para aumentar a conversão?

# **2. PREPARE**

In [1]:
import kagglehub
import os
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
# Download latest version
path = kagglehub.dataset_download("arashnic/fitbit")

print("Path to dataset files:", path)

100%|██████████| 43.3M/43.3M [00:02<00:00, 21.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2


In [3]:
# dicionário para armazenar os DataFRames agrupados por nome de arquivo
datasets = {}

# listar e tentar abrir cada csv encontrado
for root, dirs, files, in os.walk(path):
  for file in files:
    if file.endswith(".csv"): #garantir que seja CSV
      dataset_path = os.path.join(root, file)
      print(f"Lendo: {dataset_path}")
      try:
        # Leitura com tentativa de parse automático de datas
        df = pl.read_csv(dataset_path, infer_schema_length=1000)

        # Se já existe o nome de arquivo no dicinário, concatena em um único
        if file in datasets:
          datasets[file].append(df)
        else:
            datasets[file] = [df]

      except Exception as e:
        print(f"\nNão foi possível ler {file}: {e}")

# Concatena os arquivos de mesmo nome
merged_datasets = {}
for file, dfs in datasets.items():
  merged_datasets[file] = pl.concat(dfs)
  print(f"Merge concluído para {file}, linhas totais: {len(merged_datasets[file])}, \n{merged_datasets[file].schema}\n")

Lendo: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteIntensitiesNarrow_merged.csv
Lendo: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteMETsNarrow_merged.csv
Lendo: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/weightLogInfo_merged.csv
Lendo: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/hourlySteps_merged.csv
Lendo: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/heartrate_seconds_merged.csv
Lendo: /root/.cache/kagglehub/datasets/arashnic/fitbit/versions/2/mturkfitbit_export_3.12.16-4.11.16/Fitabase Data 3.12.16-4.11.16/minuteStepsNarrow_merged.csv
Lendo: /root/.cache/kagglehub/datasets/arashn

In [4]:
merged_datasets.keys()

dict_keys(['minuteIntensitiesNarrow_merged.csv', 'minuteMETsNarrow_merged.csv', 'weightLogInfo_merged.csv', 'hourlySteps_merged.csv', 'heartrate_seconds_merged.csv', 'minuteStepsNarrow_merged.csv', 'minuteSleep_merged.csv', 'hourlyIntensities_merged.csv', 'dailyActivity_merged.csv', 'minuteCaloriesNarrow_merged.csv', 'hourlyCalories_merged.csv', 'dailyCalories_merged.csv', 'minuteIntensitiesWide_merged.csv', 'sleepDay_merged.csv', 'dailyIntensities_merged.csv', 'minuteCaloriesWide_merged.csv', 'dailySteps_merged.csv', 'minuteStepsWide_merged.csv'])

In [5]:
# Preparar datasets para a análise
sleep = merged_datasets["sleepDay_merged.csv"]
heartrate = merged_datasets["heartrate_seconds_merged.csv"]
weight = merged_datasets["weightLogInfo_merged.csv"]
daily_activity = merged_datasets["dailyActivity_merged.csv"]
daily_calories = merged_datasets["dailyCalories_merged.csv"]
daily_steps = merged_datasets["dailySteps_merged.csv"]
daily_intensity = merged_datasets["dailyIntensities_merged.csv"]
hourly_intensity = merged_datasets["hourlyIntensities_merged.csv"]
hourly_calories = merged_datasets["hourlyCalories_merged.csv"]
hourly_steps = merged_datasets["hourlySteps_merged.csv"]

# **3. PROCESS**

Nesta etapa os dados serão transformados para adequação Às análises:
* Selecão nos datasets de interesse;
* Remoção de linhas duplicadas;
* Formatação e tipagem dos dados;
* Unificação dos datasets;
* Identificação de inconsistências e ruídos.
* Adição de métricas e colunas cálculadas.

In [6]:
# Criar um dicionário vinculando o nome de exibição ao DataFrame
datasets = {
    "sleep": sleep,
    "heartrate": heartrate,
    "weight": weight,
    "daily_activity": daily_activity,
    "daily_calories": daily_calories,
    "daily_steps": daily_steps,
    "daily_intensity": daily_intensity,
    "hourly_intensity": hourly_intensity,
    "hourly_calories": hourly_calories,
    "hourly_steps": hourly_steps
}

### 3.1 Remover duplicatas

In [7]:
# Cria um dicionário para armazenar os datasets após as limpezas
datasets_clean = {}

# Itera sobre o datasets para remover as linha duplicadas
for nome, df in datasets.items():
    total_antes = df.height

    # Remove duplicatas (considerando todas as colunas)
    df_clean = df.unique()

    total_depois = df_clean.height
    removidas = total_antes - total_depois

    # Armazena o DF limpo no novo dicionário
    datasets_clean[nome] = df_clean

    if removidas > 0:
        print(f"{nome}: {removidas} duplicatas removidas. (Restam {total_depois} linhas)")
    else:
        print(f"{nome}: Nenhuma duplicata encontrada.")

sleep: 3 duplicatas removidas. (Restam 410 linhas)
heartrate: 23424 duplicatas removidas. (Restam 3614915 linhas)
weight: 2 duplicatas removidas. (Restam 98 linhas)
daily_activity: Nenhuma duplicata encontrada.
daily_calories: Nenhuma duplicata encontrada.
daily_steps: Nenhuma duplicata encontrada.
daily_intensity: Nenhuma duplicata encontrada.
hourly_intensity: 175 duplicatas removidas. (Restam 46008 linhas)
hourly_calories: 175 duplicatas removidas. (Restam 46008 linhas)
hourly_steps: 175 duplicatas removidas. (Restam 46008 linhas)


### 3.2 Formatação e tipagem dos dados

In [8]:
# Iterar sobre o dicionário e converter a coluna 'Id' para String (Utf8 no Polars)
for nome, df in datasets_clean.items():
    if "Id" in df.columns:
        datasets_clean[nome] = df.with_columns(pl.col("Id").cast(pl.Utf8))

# Verificação rápida do tipo
print("Processamento concluído e dicionário atualizado.")

Processamento concluído e dicionário atualizado.


In [9]:
# Funções utilitárias para ajustar colunas de Data
def parse_date(df: pl.DataFrame, col: str, fmt: str = "%m/%d/%Y") -> pl.DataFrame:
  """Converte coluna string para Date (sem hora)."""
  return df.with_columns(
      pl.col(col).str.strip_chars().str.strptime(pl.Date, format=fmt, strict=False)
      )

def parse_datetime(df: pl.DataFrame, col: str, fmt: str = "%m/%d/%Y %I:%M:%S %p") -> pl.DataFrame:
  """Converte coluna string para Datetime (com hora AM/PM)."""
  return df.with_columns(
      pl.col(col).str.strip_chars().str.strptime(pl.Datetime, format=fmt, strict=False)
      )

In [10]:
# 1. Criar um mapeamento de qual coluna de data cada dataset usa
date_columns = {
    "sleep": ("SleepDay", "datetime"),
    "daily_activity": ("ActivityDate", "date"),
    "heartrate": ("Time", "datetime"),
    "weight": ("Date", "datetime"),
    "daily_calories": ("ActivityDay", "date"),
    "daily_steps": ("ActivityDay", "date"),
    "daily_intensity": ("ActivityDay", "date"),
    "hourly_intensity": ("ActivityHour", "datetime"),
    "hourly_calories": ("ActivityHour", "datetime"),
    "hourly_steps": ("ActivityHour", "datetime")
}

# 2. Loop para Limpar e Tipar
for nome, df in datasets_clean.items():
    # Pegar info da coluna de data
    col_nome, tipo = date_columns[nome]

    # Aplicar a conversão baseada no tipo definido no mapeamento
    if tipo == "datetime":
        df = parse_datetime(df, col_nome)
    else:
        df = parse_date(df, col_nome)

    # Salva o DF modificado de volta no dicionário
    datasets_clean[nome] = df

print("Processamento concluído e dicionário atualizado.")

Processamento concluído e dicionário atualizado.


In [11]:
# Iterar sobre o dicionário para conferir os schemas corrigidos
for nome, df in datasets_clean.items():
    print(f"{nome} {df.schema}")

sleep Schema({'Id': String, 'SleepDay': Datetime(time_unit='us', time_zone=None), 'TotalSleepRecords': Int64, 'TotalMinutesAsleep': Int64, 'TotalTimeInBed': Int64})
heartrate Schema({'Id': String, 'Time': Datetime(time_unit='us', time_zone=None), 'Value': Int64})
weight Schema({'Id': String, 'Date': Datetime(time_unit='us', time_zone=None), 'WeightKg': Float64, 'WeightPounds': Float64, 'Fat': Int64, 'BMI': Float64, 'IsManualReport': Boolean, 'LogId': Int64})
daily_activity Schema({'Id': String, 'ActivityDate': Date, 'TotalSteps': Int64, 'TotalDistance': Float64, 'TrackerDistance': Float64, 'LoggedActivitiesDistance': Float64, 'VeryActiveDistance': Float64, 'ModeratelyActiveDistance': Float64, 'LightActiveDistance': Float64, 'SedentaryActiveDistance': Float64, 'VeryActiveMinutes': Int64, 'FairlyActiveMinutes': Int64, 'LightlyActiveMinutes': Int64, 'SedentaryMinutes': Int64, 'Calories': Int64})
daily_calories Schema({'Id': String, 'ActivityDay': Date, 'Calories': Int64})
daily_steps Sche

### 3.3 Unificando datasets

**Datasets diários: Atividade e Sono**

In [12]:
# 1. Padronizar a coluna de data nos datasets diários
daily_activity = datasets_clean["daily_activity"].with_columns(pl.col("ActivityDate").alias("date"))
sleep = datasets_clean["sleep"].with_columns(pl.col("SleepDay").dt.date().alias("date"))

# 2. Join Progressivo (Left Join para não perder usuários)
df_daily = (
    daily_activity
    .join(sleep, on=["Id", "date"], how="left")
)


**Datasets por hora: Passos, Calorias e Intensidade**

In [13]:
# 1. Pegar os datasets do dicionário
h_steps = datasets_clean["hourly_steps"]
h_calories = datasets_clean["hourly_calories"]
h_intensity = datasets_clean["hourly_intensity"]

# 2. Join em ID e nas colunas de Datetime (ActivityHour)
df_hourly = h_steps.join(
    h_calories,
    on=["Id", "ActivityHour"],
    how="inner").join(
    h_intensity,
    on=["Id", "ActivityHour"],
    how="inner"
)

### 3.5 Identificar zeros e inconsistências
Nesta etapa, precisamos separar o que é "comportamento sedentário" (real) de "não uso do dispositivo" (erro de dado).

**a) Limpeza do df_daily (Foco em Zeros e Consistência)**

In [14]:
# 1. Identificar dias com 0 passos e 1440 min (24h) sedentários
# Usar flag para provável não-uso
df_daily_clean = df_daily.with_columns(
    pl.when(
        (pl.col("TotalSteps") == 0) &
        (pl.col("SedentaryMinutes") == 1440))
    .then(pl.lit(False))
    .otherwise(pl.lit(True))
    .alias("is_active_day")
)

# 2. Validar a consistência do dia (Máximo de 1440 minutos)
# Soma as 4 categorias de minutos ativos/sedentários
df_daily_clean = df_daily_clean.with_columns(
    (pl.col("VeryActiveMinutes") +
     pl.col("FairlyActiveMinutes") +
     pl.col("LightlyActiveMinutes") +
     pl.col("SedentaryMinutes")).alias("total_day_registered_minutes")
)

# Filtra para garantir apenas registros logicamente possíveis (dias com até 24h)
df_daily_clean = df_daily_clean.filter(pl.col("total_day_registered_minutes") <= 1440)



**b) Limpeza do df_hourly (foco em qualidade de uso)**

No dataset horário, o "zero" pode ser um dado real (a pessoa estava sentada), então não o removemos, apenas o sinalizamos.

In [15]:
# 1. Identificar status de atividade por hora
df_hourly_clean = df_hourly.with_columns(
    pl.when(pl.col("StepTotal") > 0)
    .then(pl.lit("Ativo"))
    .otherwise(pl.lit("Sedentário/Inativo"))
    .alias("status_hora")
)

# 2. Identificar IDs que têm pouquíssimos registros horários
# Isso ajuda a entender se o df_hourly é representativo para aquele ID
qualidade_id_horario = (
    df_hourly_clean.group_by("Id")
    .agg(pl.count("ActivityHour").alias("total_logs_horarios"))
)

### 3.4 Adicionar métricas e colunas calculadas

**1. Daily dataset**

In [16]:
# 1. Criar a Flag de Uso para o monitoramento do sono
df_daily_clean = df_daily_clean.with_columns(
    pl.col("TotalMinutesAsleep").is_not_null().alias("has_sleep_data")
)

# 2. Adiciona Atributos: dia da semana
df_daily_clean = df_daily_clean.with_columns([
    pl.col("ActivityDate").dt.weekday().alias("day_of_week") # 1=Segunda, 7=Domingo
])

# 3. Adicionar Métrica de Eficiência de Sono (Apenas para quem tem dados de sono)
df_daily_clean = df_daily_clean.with_columns(
    pl.when(pl.col("has_sleep_data") & (pl.col("TotalTimeInBed") > 0))
    .then(pl.col("TotalMinutesAsleep") / pl.col("TotalTimeInBed") * 100)
    .otherwise(None)
    .alias("eficiencia_sono")
)

**2. Hourly Dataset**

In [17]:
# 1. Adiciona Atributos: hora, dia da semana e data
df_hourly_clean = df_hourly.with_columns([
    pl.col("ActivityHour").dt.hour().alias("hour"),
    pl.col("ActivityHour").dt.weekday().alias("day_of_week"), # 1=Segunda, 7=Domingo
    pl.col("ActivityHour").dt.date().alias("date")
])

# **4. ANALYZE**

Ideias em três pilares finais:

Perfil da Usuária (Clusters): Quem são elas? (Atletas, Inconstantes, Sedentárias).

Saúde Conectada (Correlações): Como um hábito afeta o outro? (Atividade vs. Sono).

Adesão ao Ecossistema (Consistência): Elas usam todos os recursos? (Onde a Bellabeat está perdendo engajamento).

##4.1  Análise descritiva
Resumo dos principais indicadores do conjunto de dados.

In [18]:
# --- 1. Volume e Engajamento ---
total_users = df_daily_clean.select(pl.col("Id").n_unique()).item()
# % de usuárias que possuem pelo menos um registro de sono
users_with_sleep = df_daily_clean.filter(pl.col("has_sleep_data")).select(pl.col("Id").n_unique()).item()
pct_users_sleep = (users_with_sleep / total_users) * 100

# --- 2. Performance de Atividade (OMS: 10k passos) ---
avg_steps = df_daily_clean.filter(pl.col("is_active_day")).select(pl.col("TotalSteps").mean()).item()
pct_meta_reached = (df_daily_clean.filter(pl.col("TotalSteps") >= 7000).height / df_daily_clean.height) * 100

# --- 3. Composição do Dia (24h) ---
# Usa a média de quem tem sono para fechar a conta de 24h
df_sleep_avg = df_daily_clean.filter(pl.col("has_sleep_data"))
active_cols = ["VeryActiveMinutes", "FairlyActiveMinutes", "LightlyActiveMinutes"]

avg_active_hrs = df_sleep_avg.select(pl.sum_horizontal(active_cols).mean()).item() / 60
avg_sleep_hrs = df_sleep_avg.select(pl.col("TotalMinutesAsleep").mean()).item() / 60
avg_sedentary_hrs = 24 - avg_active_hrs - avg_sleep_hrs

# --- 4. Qualidade e Pico ---
avg_sleep_eff = df_daily_clean.select(pl.col("eficiencia_sono").drop_nulls().mean()).item()

# Usando o df_hourly_clean
peak_hour = (
    df_hourly_clean.group_by("hour")
    .agg(pl.col("StepTotal").mean())
    .sort("StepTotal", descending=True)
    .select(pl.col("hour").first())
    .item()
)

# Consolidação para o HTML-Cards
stats = {
    "total_users": total_users,
    "avg_steps": f"{avg_steps:,.0f}".replace(",", "."),
    "pct_meta": f"{pct_meta_reached:.1f}%",
    "peak_hour": f"{peak_hour}:00",
    "active_hrs": f"{avg_active_hrs:.1f}h",
    "sedentary_hrs": f"{avg_sedentary_hrs:.1f}h",
    "sleep_hrs": f"{avg_sleep_hrs:.1f}h",
    "sleep_eff": f"{avg_sleep_eff:.1f}%",
    "sleep_engagement": f"{pct_users_sleep:.1f}%"
}

In [19]:
from IPython.display import HTML

html_cards = f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Montserrat:wght@300;400;600&family=Playfair+Display:wght@700&display=swap');

    .dashboard-container {{
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
        gap: 25px;
        font-family: 'Montserrat', sans-serif;
        background-color: #FFFFFF; /* Fundo totalmente limpo */
        padding: 20px;
    }}

    .card {{
        background: #FFFFFF;
        padding: 25px;
        border-radius: 20px;
        /* Sombra muito suave para não pesar */
        box-shadow: 0 10px 30px rgba(0,0,0,0.04);
        border: 1px solid #F1F1F1;
        transition: all 0.3s ease;
        text-align: center; /* Centralizado para um look mais app-design */
    }}

    .card:hover {{
        transform: translateY(-5px);
        box-shadow: 0 15px 35px rgba(157, 128, 179, 0.1);
        border-color: #9D80B3;
    }}

    .card-title {{
        font-size: 0.7rem;
        color: #A0A0A0;
        text-transform: uppercase;
        letter-spacing: 2px;
        margin-bottom: 15px;
        font-weight: 600;
    }}

    .card-value {{
        font-family: 'Montserrat', serif;
        font-size: 2.4rem;
        font-weight: 700;
        margin-bottom: 8px;
    }}

    /* Cores das métricas baseadas na sua sugestão */
    .color-peach {{ color: #F9AD94; }}
    .color-lavender {{ color: #9D80B3; }}
    .color-light-green {{ color: #AFDF9B; }}

    .card-subtitle {{
        font-size: 0.8rem;
        color: #B0B0B0;
        font-weight: 400;
        border-top: 1px solid #F8F8F8;
        padding-top: 12px;
        margin-top: 12px;
    }}

    .badge {{
        background: #FFF5F2;
        color: #F9AD94;
        padding: 3px 8px;
        border-radius: 12px;
        font-size: 0.7rem;
        font-weight: 600;
    }}
</style>

<div class="dashboard-container">
    <div class="card">
        <div class="card-title">Usuárias Ativas</div>
        <div class="card-value color-lavender">{stats['total_users']}</div>
        <div class="card-subtitle">Base total monitorada</div>
    </div>

    <div class="card">
        <div class="card-title">Média de Passos</div>
        <div class="card-value color-peach">{stats['avg_steps']}</div>
        <div class="card-subtitle"><span class="badge">{stats['pct_meta']}</span> meta 7k</div>
    </div>

    <div class="card">
        <div class="card-title">Pico de Energia</div>
        <div class="card-value color-light-green">{stats['peak_hour']}</div>
        <div class="card-subtitle">Horário de maior esforço</div>
    </div>

    <div class="card">
        <div class="card-title">Tempo de Sono</div>
        <div class="card-value color-lavender">{stats['sleep_hrs']}</div>
        <div class="card-subtitle">Eficiência média: <span style="color:#9D80B3">{stats['sleep_eff']}</span></div>
    </div>

    <div class="card">
        <div class="card-title">Tempo Ativo</div>
        <div class="card-value color-peach">{stats['active_hrs']}</div>
        <div class="card-subtitle">Média em movimento</div>
    </div>

    <div class="card">
        <div class="card-title">Sedentarismo</div>
        <div class="card-value color-light-green">{stats['sedentary_hrs']}</div>
        <div class="card-subtitle">Foco em bem-estar</div>
    </div>
</div>
"""

HTML(html_cards)

## 4.2 Perfil de usuários (segmentação)


Para entender quem são os usuários vamos agrupa-los em  clusters, com o KMeans, partindo do tempo de cada intensidade nas atividades - "Sedentary", "Lightly", "Fairly Active" e "Very Active".
Ao final serão três categorias:
  * Sedentários: muitos minutos sedentários;
  * Moderados: boa quantidade de minutos em atividades leves;
  * Intensos: muitos minutos em atividades bastante e/ou muito ativas.

In [20]:
# Preparação dos Dados
# adição de novas colunas cálculadas para segmentação dos dados
df_weighted = (
    df_daily_clean
    .with_columns([
        ((pl.col("VeryActiveMinutes") + pl.col("FairlyActiveMinutes")) / pl.col("total_day_registered_minutes") * 100).alias("pct_intense"),
        (pl.col("LightlyActiveMinutes") / pl.col("total_day_registered_minutes") * 100).alias("pct_lightly_active"),
        (pl.col("SedentaryMinutes") / pl.col("total_day_registered_minutes") * 100).alias("pct_sedentary")
    ])
    .group_by("Id")
    .agg([
        pl.col("pct_sedentary").mean().alias("avg_pct_sedentary"),
        pl.col("pct_lightly_active").mean().alias("avg_pct_lightly_active"),
        pl.col("pct_intense").mean().alias("avg_pct_intense"),
        pl.col("TotalSteps").mean().alias("avg_steps"),
        pl.col("date").n_unique().alias("days_active")
    ])
    .filter(pl.col("days_active") >= 7)
    .drop_nulls()
)


In [21]:
# --- KMEANS ---
# 1. Preparação e Fit
scaler = StandardScaler()
features = ["avg_pct_sedentary", "avg_pct_lightly_active", "avg_pct_intense"]
scaled_data = scaler.fit_transform(df_weighted.select(features).to_numpy())

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(scaled_data)

# Adiciona o cluster numérico temporário
df_weighted = df_weighted.with_columns(pl.Series("cluster_temp", clusters))

# 2. Lógica de Identificação Automática
# Calcula a média de 'intensidade' para cada ID de cluster
centroids = kmeans.cluster_centers_
# 'avg_pct_intense' é a 3ª coluna (índice 2) do scaler
intense_idx = 2
avg_intensities = centroids[:, intense_idx]

# Cria um ranking: quem tem a menor média de intensidade é 'Sedentário'
# quem tem a maior é 'Intenso'
order = np.argsort(avg_intensities)

# Mapeamento dinâmico baseado nos dados
# order[0] -> menor intensidade (Sedentários)
# order[1] -> média (Moderados)
# order[2] -> maior intensidade (Intensos)
dynamic_map = {
    str(order[0]): "Sedentários",
    str(order[1]): "Moderados",
    str(order[2]): "Intensos"
}

# 3. Mapeamento Final
df_viz = df_weighted.to_pandas()
df_viz['perfil_nome'] = df_viz['cluster_temp'].astype(str).map(dynamic_map)

# Cria um df de referência apenas com ID e o Perfil já classificado
df_perfis = pl.from_pandas(df_viz[['Id', 'perfil_nome']])

In [22]:
# --- CONFIGURAÇÕES DE DESIGN (ESTILO) ---
PALETA_CORES = ['#B6A0C6', '#F9AD94', '#AFDF9B']
CATEGORIAS = ['Sedentários', 'Moderados', 'Intensos']
TEXTO_PERCENTUAL = "datum.value + '%'"

# Configurações de Eixo e Títulos
estilo_eixo = alt.Axis(gridOpacity=0.4, labelExpr=TEXTO_PERCENTUAL)
config_view = {"strokeWidth": 0}
estilo_fonte = {"labelFontSize": 12, "titleFontSize": 14}

# --- DEFINIÇÃO DOS COMPONENTES ---

# Escalas e Cores
color_scale = alt.Color('perfil_nome:N',
    title='Perfil de Usuária',
    scale=alt.Scale(domain=CATEGORIAS, range=PALETA_CORES),
    legend=alt.Legend(padding=10, strokeColor='#ccc', cornerRadius=5, fillColor='white')
)

# Tooltips reutilizáveis
tooltips = [
    alt.Tooltip('Id:N', title='ID'),
    alt.Tooltip('avg_pct_sedentary:Q', title='% Sedentário', format='.1f'),
    alt.Tooltip('avg_pct_lightly_active:Q', title='% Leve', format='.1f'),
    alt.Tooltip('avg_pct_intense:Q', title='% Intensa', format='.1f'),
    alt.Tooltip('avg_steps:Q', title='Média de Passos', format=',.0f')
]

# --- CONSTRUÇÃO DO GRÁFICO ---

# Base compartilhada (Eixo X fixo)
base = alt.Chart(df_viz).mark_circle(size=200, opacity=0.7, stroke='white', strokeWidth=1).encode(
    x=alt.X('avg_pct_sedentary:Q',
            title='Tempo Sedentário (%)',
            scale=alt.Scale(domain=[df_viz['avg_pct_sedentary'].min() - 5, 100]),
            axis=estilo_eixo),
    color=color_scale,
    tooltip=tooltips
).properties(width=320, height=450)

# Gráfico Leve
chart_leve = base.encode(
    y=alt.Y('avg_pct_lightly_active:Q',
            title='Atividade Leve (%)',
            scale=alt.Scale(domain=[0, df_viz['avg_pct_lightly_active'].max() + 5]),
            axis=estilo_eixo)
).properties(title="Volume de Atividade Leve")

# Gráfico Intenso
chart_intensa = base.encode(
    y=alt.Y('avg_pct_intense:Q',
            title='Atividade Intensa (%)',
            scale=alt.Scale(domain=[0, df_viz['avg_pct_intense'].max() + 2]),
            axis=estilo_eixo)
).properties(title="Volume de Atividade Intensa")

# --- LAYOUT FINAL ---

final_chart = alt.hconcat(chart_leve, chart_intensa).resolve_scale(
    x='shared',
    color='shared'
).properties(
    title={
        "text": "Impacto do Sedentarismo na Atividade Diária",
        "subtitle": ["Análise de como o aumento do tempo sedentário reduz as janelas de atividade."],
        "fontSize": 22, "subtitleFontSize": 14, "anchor": "middle", "offset": 20
    }
).configure_view(
    **config_view
).configure_axis(
    **estilo_fonte
).interactive()

final_chart.display()

alt.HConcatChart(...)

A seguir vamos colocar esses mesmos perfis em outra perspectiva: qual a variância da intensidade das atividades?

In [23]:
# 1. Preparação do df_user (Agregando dados por hora)
df_user = df_hourly_clean.group_by("Id").agg([
    pl.col("TotalIntensity").mean().alias("mean"),
    pl.col("TotalIntensity").std().alias("std"),
    pl.col("ActivityHour").dt.date().n_unique().alias("days_active")
]).filter(pl.col("days_active") >= 7).drop_nulls()

# 2. Join com os perfis já determinados
df_user_final = df_user.join(df_perfis, on="Id", how="left")

# Convertendo para Pandas para o Altair
df_viz = df_user_final.to_pandas()

# 3. Gráfico Interativo
cluster_chart = alt.Chart(df_viz).mark_circle(size=100).encode(
    x=alt.X('mean:Q', title='Média de Intensidade por Hora'),
    y=alt.Y('std:Q', title='Variabilidade (Desvio Padrão)'),
    color=alt.Color('perfil_nome:N',
                    scale=alt.Scale(
                        domain=['Sedentários', 'Moderados', 'Intensos'],
                        range=['#B6A0C6', '#F9AD94', '#AFDF9B']),
                    title='Perfil de Usuária',
                    sort=['Sedentários', 'Moderados', 'Intensos']), # Força a ordem na legenda
    tooltip=[
        alt.Tooltip('Id:N', title='ID da Usuária'),
        alt.Tooltip('mean:Q', title='Média', format='.1f'),
        alt.Tooltip('std:Q', title='Consistência (Desvio)', format='.1f'),
        alt.Tooltip('days_active:Q', title='Dias Ativos'),
        alt.Tooltip('perfil_nome:N', title='Perfil')
    ]
).properties(
    title='Segmentação Bellabeat: Performance vs Consistência (Baseado no Perfil de Intensidade)',
    width=600,
    height=400
).interactive()

cluster_chart.display()

alt.Chart(...)

##4.3 Correlação entre atividade e sono
Usuários que caminham mais tendem a dormir melhor?

In [24]:
# 1. Ajuste no Processamento (Convertendo minutos para horas)
df_final = (
    df_daily_clean
    .filter(pl.col("has_sleep_data"))
    .join(df_perfis, on="Id")
    .group_by("perfil_nome")
    .agg([
        (pl.col("TotalMinutesAsleep").mean() / 60).alias("sono_horas"), # Conversão para Horas
        (pl.col("eficiencia_sono").mean()).alias("eficiencia")
    ])
    .to_pandas()
)

# Configuração Base
base = alt.Chart(df_final).encode(
    x=alt.X('perfil_nome:N', title=None, sort='-y', axis=alt.Axis(labelAngle=0)),
    color=alt.Color('perfil_nome:N', legend=None,
                    scale=alt.Scale(domain=['Sedentários', 'Moderados', 'Intensos'],
                                    range=['#B6A0C6', '#F9AD94', '#AFDF9B']))
).properties(width=200, height=250)

# 2. Gráfico de Horas de Sono
bars_sono = base.mark_bar(cornerRadiusEnd=5).encode(
    y=alt.Y('sono_horas:Q', title='Horas por Noite', scale=alt.Scale(domain=[0, 9]))
)

text_sono = bars_sono.mark_text(align='center', baseline='bottom', dy=-5, fontWeight='bold').encode(
    text=alt.Text('sono_horas:Q', format='.1f')
)

chart_sono = (bars_sono + text_sono).properties(title="Duração Média")

# 3. Gráfico de Eficiência (Escala completa para evitar distorção)
bars_efici = base.mark_bar(cornerRadiusEnd=5).encode(
    y=alt.Y('eficiencia:Q', title='Eficiência (%)', scale=alt.Scale(domain=[0, 100]))
)

text_efici = bars_efici.mark_text(align='center', baseline='bottom', dy=-5, fontWeight='bold').encode(
    text=alt.Text('eficiencia:Q', format='.0f')
)

chart_efici = (bars_efici + text_efici).properties(title="Qualidade (Eficiência)")

# Exibição Combinada
alt.hconcat(chart_sono, chart_efici).configure_view(
    strokeWidth=0
).configure_axis(
    grid=False,
    domain=False
).properties(
    title=alt.TitleParams(
        text="Perfis Moderados apresentam maior qualidade de sono",
        subtitle=["Média de horas dormidas e eficiência por categoria de usuário"],
        fontSize=20,
        anchor='middle',
        subtitleFontSize=14,
        dy=-20
    )
).display()

alt.HConcatChart(...)

**Visualização Interativa: Eficiência do Sono vs. Sedentarismo**

O objetivo aqui é ver se usuárias que passam muito tempo sedentárias durante o dia têm mais dificuldade para pegar no sono (mais tempo acordada na cama).

In [25]:
# 1. Preparação dos Dados (Conversão para Horas e Limpeza)
df_bins = (
    df_daily_clean
    .filter((pl.col("has_sleep_data")) & (pl.col("TotalMinutesAsleep") > 120))
    .join(df_perfis.select(["Id", "perfil_nome"]), on="Id")
    .with_columns([
        (pl.col("TotalMinutesAsleep") / 60).round(2).alias("sono_horas"),
        pl.col("SedentaryMinutes").qcut(4, labels=["Baixo", "Razoável", "Moderado", "Alto"])
          .alias("faixa_sedentarismo")
    ])
    .to_pandas()
)

# 2. Definições de Estilo
palette = ['#AFDF9B', '#F9AD94', '#B6A0C6']
perfil_order = ['Intensos', 'Moderados', 'Sedentários']
sedentario_order = ["Baixo", "Razoável", "Moderado", "Alto"]

# 3. Construção do Gráfico
chart = alt.Chart(df_bins).mark_boxplot(
    extent='min-max',
    ticks=True,
    size=30,
    outliers=True
).encode(
    x=alt.X('faixa_sedentarismo:N',
            title=None,
            sort=sedentario_order,
            axis=alt.Axis(labelAngle=0, labelFontSize=11, labelPadding=10)),
    y=alt.Y('sono_horas:Q',
            title="Horas Dormidas",
            scale=alt.Scale(domain=[0, 12])),
    color=alt.Color('perfil_nome:N',
                    scale=alt.Scale(domain=perfil_order, range=palette),
                    legend=None)
).properties(
    width=220,
    height=280
).facet(
    column=alt.Column('perfil_nome:N',
                      sort=perfil_order,
                      title=None,
                      header=alt.Header(labelFontSize=14, labelFontWeight='bold'))
).resolve_scale(
    y='shared'
)

# 4. Ajustes Finais de Layout
final_chart = chart.properties(
    title=alt.TitleParams(
        text="Impacto do Sedentarismo na Duração do Sono",
        subtitle=["Distribuição do tempo de sono conforme o nível de inatividade diária"],
        anchor='middle',
        fontSize=20,
        subtitleFontSize=14,
        dy=-20
    )
).configure_view(
    strokeWidth=0
).configure_axis(
    grid=False,
    domain=False,
    tickSize=0
)

final_chart.display()

alt.FacetChart(...)

In [26]:
# 1. Preparação dos Dados
df_sleep_refined = (
    df_daily_clean
    .filter((pl.col("has_sleep_data")) & (pl.col("TotalMinutesAsleep") > 120))
    .join(df_perfis.select(["Id", "perfil_nome"]), on="Id")
    .group_by(["Id", "perfil_nome"])
    .agg([
        (pl.col("TotalMinutesAsleep").mean() / 60).alias("avg_sleep_h"),
        (pl.col("TotalTimeInBed").mean() / 60).alias("avg_bed_h"),
        pl.col("eficiencia_sono").mean().alias("avg_efficiency"),
        pl.col("date").n_unique().alias("days")
    ])
    .filter(pl.col("days") >= 7)
    .with_columns([
        pl.col("avg_sleep_h").round(2),
        pl.col("avg_bed_h").round(2)
    ])
    .to_pandas()
)

# 2. Estilo e Escalas
palette = ['#AFDF9B', '#F9AD94', '#B6A0C6']
perfil_order = ['Intensos', 'Moderados', 'Sedentários']
min_focus = 4
max_val = df_sleep_refined[["avg_sleep_h", "avg_bed_h"]].max().max()
limit_max = max_val * 1.05
shared_scale = alt.Scale(domain=[min_focus, limit_max], zero=False, nice=True)

# 3. Construção do Gráfico
base = alt.Chart(df_sleep_refined).encode(
    x=alt.X('avg_sleep_h:Q',
            title='Horas Dormindo',
            scale=shared_scale,
            axis=alt.Axis(tickCount=5, labelPadding=10)),
    y=alt.Y('avg_bed_h:Q',
            title='Horas na Cama',
            scale=shared_scale,
            axis=alt.Axis(tickCount=5, labelPadding=10))
)

# Camada 1: Linha de Referência (Eficiência 100%)
line = alt.Chart(pd.DataFrame({'val': [min_focus, limit_max]})).mark_line(
    color='#D1D1D1',
    strokeDash=[4,4],
    strokeWidth=1.5
).encode(x='val:Q', y='val:Q')

# Camada 2: Pontos
points = base.mark_circle(size=130, opacity=0.7, stroke='white', strokeWidth=0.5).encode(
    color=alt.Color('perfil_nome:N',
                    title=None,
                    scale=alt.Scale(domain=perfil_order, range=palette),
                    legend=alt.Legend(orient='top-left', padding=10, symbolSize=100)),
    tooltip=[
        alt.Tooltip('perfil_nome:N', title='Perfil'),
        alt.Tooltip('avg_sleep_h:Q', title='Sono (h)', format='.1f'),
        alt.Tooltip('avg_bed_h:Q', title='Na Cama (h)', format='.1f'),
        alt.Tooltip('avg_efficiency:Q', title='Eficiência (%)', format='.2f')
    ]
)

# 4. Ajustes Finais
final_chart = (line + points).properties(
    width=500, height=300,
    title={
        "text": "Qualidade do Repouso: Sono vs. Tempo na Cama",
        "subtitle": [
            "A proximidade com a linha indica maior eficiência (menos tempo acordado)."
        ],
        "anchor": "middle", "fontSize": 20, "subtitleFontSize": 14, "dy": -15
    }
).configure_axis(
    grid=True,
    gridColor='#F5F5F5',
    domain=False,
    tickSize=0
).configure_view(
    strokeWidth=0
)

final_chart.display()

alt.LayerChart(...)

##4.4 Horários de atividade
Insight: campanhas podem sugerir treinos curtos nos horários em que usuários já são mais ativos.

In [27]:
# 1.Calcular a média de passos por dia da semana e hora
df_heat_agg = (
    df_hourly_clean
    .group_by(["day_of_week", "hour"])
    .agg(pl.col("StepTotal").mean().alias("avg_steps"))
    .sort(["day_of_week", "hour"])
    .to_pandas()
)

# 2. Mapear os nomes dos dias (Atenção: Polars usa 1=Seg, 7=Dom)
dias_map = {1: "Seg", 2: "Ter", 3: "Qua", 4: "Qui", 5: "Sex", 6: "Sáb", 7: "Dom"}
df_heat_agg['day_name'] = df_heat_agg['day_of_week'].map(dias_map)

# 3. Criar o Heatmap no Altair
heatmap = alt.Chart(df_heat_agg).mark_rect().encode(
    x=alt.X('day_name:O',
            sort=["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"],
            title="Dia da Semana"),
    y=alt.Y('hour:O', title="Hora do Dia"),
    color=alt.Color('avg_steps:Q',
                    scale=alt.Scale(scheme='purplered'),
                    title="Passos"),
    tooltip=[
        alt.Tooltip('day_name:N', title='Dia'),
        alt.Tooltip('hour:O', title='Hora'),
        alt.Tooltip('avg_steps:Q', title='Média', format='.0f')
    ]
).properties(
    title={
        "text": "Horários de Maior Atividade das Usuárias",
        "subtitle": "Média de passos por hora e dia da semana"
    },
    width=500,
    height=400
).configure_axis(
    labelFontSize=12,
    titleFontSize=14
)

heatmap.display()

alt.Chart(...)

In [28]:
# 1. Preparação dos Dados
dias_map = {1: "Seg", 2: "Ter", 3: "Qua", 4: "Qui", 5: "Sex", 6: "Sáb", 7: "Dom"}

df_heat_faceted = (
    df_hourly_clean
    .join(df_perfis.select(["Id", "perfil_nome"]), on="Id")
    .group_by(["perfil_nome", "day_of_week", "hour"])
    .agg(pl.col("StepTotal").mean().alias("avg_steps"))
    .with_columns(pl.col("day_of_week").cast(pl.String).replace(dias_map).alias("day_name"))
    .to_pandas()
)

# 2. Constantes de Ordenação
perfil_order = ['Intensos', 'Moderados', 'Sedentários']
dias_order = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]

# 3. Gráfico Facetado
heatmap = alt.Chart(df_heat_faceted).mark_rect().encode(
    x=alt.X('day_name:O', sort=dias_order, title=None, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('hour:O', title="Hora"),
    color=alt.Color('avg_steps:Q', title="Passos", scale=alt.Scale(scheme='purplered')),
    tooltip=[
        alt.Tooltip('perfil_nome:N', title='Perfil'),
        alt.Tooltip('day_name:N', title='Dia'),
        alt.Tooltip('hour:O', title='Hora'),
        alt.Tooltip('avg_steps:Q', title='Média', format=',.0f')
    ]
).properties(
    width=200, height=280
).facet(
    column=alt.Column('perfil_nome:N', sort=perfil_order, title=None)
).resolve_scale(
    color='independent'
)

# 4. Configurações Globais (Estilização em Bloco)
final_chart = heatmap.properties(
    title={
        "text": "Padrões de Atividade: Hora x Dia da Semana",
        "subtitle": ["Comparação de rotinas semanais entre os perfis de usuários"],
        "fontSize": 20, "anchor": "middle", "offset": 20, "subtitleFontSize": 14}
).configure_view(
    strokeWidth=0
).configure_axis(
    domain=False, ticks=False, labelFontSize=11
).configure_header(
    labelFontSize=14, labelFontWeight='bold'
)

final_chart.display()

alt.FacetChart(...)

In [29]:
# 1. Preparar os dados (Direto e ordenado)
usage_by_hour = (
    df_hourly_clean
    .group_by("hour")
    .agg(pl.col("TotalIntensity").mean())
    .sort("hour")
    .to_pandas()
)

# 2. Definição da Cor
main_color = '#AFDF9B'

# 3. Gráfico de Área Estilizado
area_chart = alt.Chart(usage_by_hour).mark_area(
    line={'color': main_color},
    point={'fill': main_color, 'stroke': main_color, 'size': 15},
    color=alt.Gradient(
        gradient='linear',
        stops=[alt.GradientStop(color='white', offset=0),
               alt.GradientStop(color=main_color, offset=1)],
        x1=1, x2=1, y1=1, y2=0
    )
).encode(
    x=alt.X('hour:O', title='Hora', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('TotalIntensity:Q', title='Intensidade'),
    tooltip=[
        alt.Tooltip('hour:O', title='Hora'),
        alt.Tooltip('TotalIntensity:Q', title='Intensidade', format='.1f')
    ]
).properties(
    title={
        "text": "Distribuição da Atividade Física ao Longo do Dia",
        "subtitle": "Média de intensidade total por hora (todos os usuários)",
    "fontSize": 20, "anchor": "middle", "subtitleFontSize": 14, "offset": 20
        },
    width=600, height=300
)

area_chart.display()

alt.Chart(...)

In [30]:
# 1. Preparação dos Dados (Agregação Expressiva)
df_heatmap_refined = (
    df_hourly_clean
    .join(df_perfis.select(["Id", "perfil_nome"]), on="Id")
    .group_by([pl.col("ActivityHour").dt.hour().alias("hour"), "perfil_nome"])
    .agg(pl.col("TotalIntensity").mean().alias("avg"))
    .to_pandas()
)

# 2. Definições de Estilo
perfil_order = ['Intensos', 'Moderados', 'Sedentários']
cores = ['#AFDF9B', '#F9AD94', '#B6A0C6']

# 3. Gráfico Simplificado
chart = alt.Chart(df_heatmap_refined).mark_line(
    size=3, interpolate='monotone', point=True
).encode(
    x=alt.X('hour:Q', title='Hora do Dia', scale=alt.Scale(domain=[0, 23])),
    y=alt.Y('avg:Q', title='Intensidade Média'),
    color=alt.Color('perfil_nome:N', sort=perfil_order,
                    scale=alt.Scale(domain=perfil_order, range=cores),
                    legend=alt.Legend(title="Segmento")),
    tooltip=[
        alt.Tooltip('hour:Q', title='Hora'),
        alt.Tooltip('perfil_nome:N', title='Perfil'),
        alt.Tooltip('avg:Q', title='Média', format='.1f')
    ]
).properties(
    title={"text": "Intensidade de Atividade por Hora", "subtitle": "Evolução horária por perfil",
           "fontSize": 20, "anchor": "middle", "subtitleFontSize": 14, "offset": 20},
    width=600, height=400
).interactive()

chart.display()

alt.Chart(...)

## 4.5 Retenção de usuárias

Consistência entre Datasets (Uso dos Produtos)
* Por quê: Se descobrir que 100% das usuárias registram passos, mas apenas 40% registram sono e 10% registram peso, está identificado um problema de produto.

* Ação: A Bellabeat precisa facilitar o registro de peso ou tornar o relógio mais confortável para dormir. Isso responde diretamente à pergunta do case: "Quais são as tendências no uso de dispositivos inteligentes?"

In [31]:
# Iterar sobre o dicionário para verificar número único de Ids
for nome, df in datasets_clean.items():
    print(f"{nome}: {df['Id'].n_unique()} IDs únicos.")

sleep: 24 IDs únicos.
heartrate: 15 IDs únicos.
weight: 13 IDs únicos.
daily_activity: 35 IDs únicos.
daily_calories: 33 IDs únicos.
daily_steps: 33 IDs únicos.
daily_intensity: 33 IDs únicos.
hourly_intensity: 35 IDs únicos.
hourly_calories: 35 IDs únicos.
hourly_steps: 35 IDs únicos.


In [32]:
# 1. Preparação dos dados
dias_pt = {1: "Segunda", 2: "Terça", 3: "Quarta", 4: "Quinta", 5: "Sexta", 6: "Sábado", 7: "Domingo"}

df_engagement = (
    df_daily_clean
    .group_by("date")
    .agg([
        pl.col("Id").n_unique().alias("Atividade Geral"),
        pl.col("Id").filter(pl.col("has_sleep_data") == True).n_unique().alias("Registro de Sono"),
        pl.col("day_of_week").first().alias("dow")
    ])
    .with_columns(
        pl.col("dow").cast(pl.String).replace(dias_pt).alias("dia_semana")
    )
    .to_pandas()
    .melt(
        id_vars=['date', 'dia_semana'],
        value_vars=['Atividade Geral', 'Registro de Sono'],
        var_name='Métrica',
        value_name='Usuárias'
    )
)

# 2. Gráfico
chart = alt.Chart(df_engagement).mark_line(
    point={'size': 30},
    strokeWidth=3,
    interpolate='monotone'
).encode(
    x=alt.X('date:T', title=None, axis=alt.Axis(format='%d/%m', labelAngle=-45)),
    y=alt.Y('Usuárias:Q', title='Usuárias Ativas'),
    color=alt.Color('Métrica:N', scale=alt.Scale(range=['#FFAC89', '#B6A0C6'])),
    tooltip=[
        alt.Tooltip('date:T', title='Data', format='%d/%m'),
        alt.Tooltip('dia_semana:N', title='Dia'),
        alt.Tooltip('Métrica:N'),
        alt.Tooltip('Usuárias:Q')
    ]
).properties(
    title={
        "text": "Evolução Diária de Engajamento",
        "subtitle": "Comparativo diário entre uso do App e registros de sono",
        "fontSize": 20, "anchor": "middle", "offset": 20,'subtitleFontSize': 14
    },
    width=650, height=350
)

chart.display()

alt.Chart(...)

In [39]:
# 1. Preparação dos Dados
dias_pt = {1: "Segunda", 2: "Terça", 3: "Quarta", 4: "Quinta", 5: "Sexta", 6: "Sábado", 7: "Domingo"}
total_usuarios = df_daily_clean.select(pl.col("Id").n_unique()).item()

# 2. Processamento com Polars
df_base_diaria = (
    df_daily_clean
    .group_by("date")
    .agg([
        pl.col("Id").n_unique().alias("Ativas"),
        pl.col("Id").filter(pl.col("has_sleep_data") == True).n_unique().alias("Sono"),
        pl.col("day_of_week").first().alias("dow")
    ])
    .sort("date")
)

# Aplicamos as lógicas distintas de cada gráfico
df_final = (
    df_base_diaria
    .with_columns([
        # Lógica do Código 1: Uso Geral baseado no acumulado (cum_max)
        (pl.col("Ativas") / pl.col("Ativas").cum_max()).alias("Uso Geral"),
        # Lógica do Código 2: Sono baseado no total real de usuários
        (pl.col("Sono") / total_usuarios).alias("Registro de Sono")
    ])
    .group_by("dow")
    .agg([
        # Média normal para Uso Geral
        pl.col("Uso Geral").mean(),
        # Média filtrada para Registro de Sono (remove dias zerados/meses de atraso)
        pl.col("Registro de Sono").filter(pl.col("Registro de Sono") > 0).mean()
    ])
    .with_columns(
        pl.col("dow").cast(pl.String).replace(dias_pt).alias("dia")
    )
    .to_pandas()
    .melt(
        id_vars=["dia"],
        value_vars=["Uso Geral", "Registro de Sono"],
        var_name="Metrica",
        value_name="Taxa"
    )
)

# 3. Configurações Visuais do Altair
ordem_dias = list(dias_pt.values())
cores = alt.Scale(domain=["Uso Geral", "Registro de Sono"], range=["#F9AD94", "#B6A0C6"])

chart_proporcao = alt.Chart(df_final).mark_bar(
    cornerRadiusTopLeft=5,
    cornerRadiusTopRight=5
).encode(
    x=alt.X('dia:N', sort=ordem_dias, title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('Taxa:Q', title="Proporção da Base", axis=alt.Axis(format='%'), scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('Metrica:N', scale=cores, legend=None),
    tooltip=[
        alt.Tooltip('dia:N', title='Dia'),
        alt.Tooltip('Metrica:N', title='Tipo'),
        alt.Tooltip('Taxa:Q', title='Proporção', format='.1%')
    ]
).properties(
    width=300, height=280
).facet(
    column=alt.Column('Metrica:N', title=None)
).resolve_scale(
    y='shared'
)

# 4. Estilo Final
final_proporcao = chart_proporcao.properties(
    title={
        "text": "Fidelidade e Retenção da Base",
        "subtitle": "Análise de Engajamento: Retenção de Uso Geral vs. Adoção do Registro de Sono",
        "fontSize": 20, "anchor": "middle", "offset": 20, 'subtitleFontSize': 14
    }
).configure_view(stroke=None).configure_axis(grid=False)

final_proporcao.display()

alt.FacetChart(...)

In [35]:
# 1. Processamento de Dados
dias_pt = {1: "Segunda", 2: "Terça", 3: "Quarta", 4: "Quinta", 5: "Sexta", 6: "Sábado", 7: "Domingo"}

df_semanal = (
    df_daily_clean
    .group_by("date")
    .agg([
        pl.col("Id").n_unique().alias("Usuárias Ativas"),
        pl.col("Id").filter(pl.col("has_sleep_data") == True).n_unique().alias("Registros de Sono"),
        pl.col("day_of_week").first().alias("dow")
    ])
    .group_by("dow")
    .agg([
        pl.col("Usuárias Ativas").mean(),
        pl.col("Registros de Sono").filter(pl.col("Registros de Sono") > 0).mean().alias("Registros de Sono")
    ])
    .with_columns(pl.col("dow").cast(pl.String).replace(dias_pt).alias("dia"))
    .to_pandas()
    .melt(id_vars=["dia"], value_vars=["Usuárias Ativas", "Registros de Sono"], var_name="tipo", value_name="media")
)

# 3. Configurações Visuais do Altair
ordem_dias = list(dias_pt.values())
cores = alt.Scale(domain=["Usuárias Ativas", "Registros de Sono"], range=["#F9AD94", "#B6A0C6"])

chart = alt.Chart(df_semanal).mark_bar(
    cornerRadiusTopLeft=5, cornerRadiusTopRight=5
).encode(
    x=alt.X('dia:N', sort=ordem_dias, title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('media:Q', title="Média de Usuárias (Dias Ativos)"),
    color=alt.Color('tipo:N', scale=cores, legend=None),
    tooltip=[
        alt.Tooltip('dia:N', title='Dia'),
        alt.Tooltip('tipo:N', title='Métrica'),
        alt.Tooltip('media:Q', title='Média Real', format='.1f')
    ]
).properties(
    width=280, height=250
).facet(
    column=alt.Column('tipo:N', title=None, header=alt.Header(labelFontSize=14, labelFontWeight='bold'))
).resolve_scale(
    y='shared'
)

final_chart = chart.properties(
    title={
        "text": "Média de Engajamento por Dia da Semana",
        "subtitle": "Comparativo entre uso geral do App e registros de sono",
        "fontSize": 20, "anchor": "middle", "offset": 20, 'subtitleFontSize': 14
    }
).configure_view(stroke=None).configure_axis(grid=False)

final_chart.display()

alt.FacetChart(...)